# Cleaning RIPA Datasets
Based off of https://github.com/joshuagrossman/ripa/tree/main/src/ripa cleaning scripts

In [ ]:
import requests # To download files from the internet
import zipfile # To open zip files
import io # To treat downloaded data like a file in memory
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
def load_ripa_from_doj_zip(county):
    """
    Downloads DOJ RIPA Stop Data (2019–2023),
    loads only the Excel file that contains the county name,
    and returns a combined DataFrame.
    
    Example:
        df = load_ripa_from_doj_zip("Orange")
    """

    # DELETE LATER
    # url_dict = {2020: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2020.zip"}

    # DOJ public ZIP URLs for 2019–2023
    url_dict = {
        2019: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2019.zip",
        2020: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2020.zip",
        2021: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2021.zip",
        2022: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-05/RIPA-Stop-Data-2022.zip",
        2023: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-05/RIPA-Stop-Data-2023.zip",
    }

    all_years = []

    for year, zip_url in url_dict.items():

        response = requests.get(zip_url)
        response.raise_for_status() # Checks whether website request succeeded

        z = zipfile.ZipFile(io.BytesIO(response.content)) # Opens the zip in memory

        # Find the Excel file containing the county name
        county_file = None # Creates placeholder value
        for file in z.namelist(): # For every file in the ZIP
            if county.lower() in file.lower() and file.lower().endswith(".xlsx"): # Check if file is the right county
                county_file = file
                break

        # If there's no file for that county
        if county_file is None:
            raise ValueError(f"No file found for {county} in {year}")

        # Open the file
        with z.open(county_file) as f:
            df_year = pd.read_excel(f, engine="openpyxl")

        # Make all columns lower-case strings
        df_year.columns = df_year.columns.str.lower()
        df_year["year"] = year # Add year column

        all_years.append(df_year)
        print(f"{year} loaded successfully.")

    final_df = pd.concat(all_years, ignore_index=True)

    return final_df


In [3]:
def add_derived_vars(df):
    """
    Adds the following derived variables:
      - race_ethnicity
      - gender
      - reason_for_contact
      - suspicion
      - action_any_search
      - search_basis_*
      - contraband_weapons
      - contraband_any
      - result_of_stop_arrest

    Returns a new DataFrame (copy) with added columns.
    """

    df = df.copy()

    # ------------------------------------------------------------
    # RACE/ETHNICITY
    # For each row, assign their corresponding race in a new column based on indicator columns
    # Ordered specifically so Hispanic/Latino comes before White, so if both races, labeled Hispanic/Latino
    # ------------------------------------------------------------
    
    df["race_ethnicity"] = np.select(
        [
            df["rae_hispanic_latino"] == 1,
            df["rae_black_african_american"] == 1,
            df["rae_asian"] == 1,
            df["rae_middle_eastern_south_asian"] == 1,
            df["rae_pacific_islander"] == 1,
            df["rae_native_american"] == 1,
            df["rae_white"] == 1,
        ],
        [
            "Hispanic",
            "Black",
            "Asian",
            "Middle Eastern/South Asian",
            "Pacific Islander",
            "Native American",
            "White",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # GENDER
    # ------------------------------------------------------------

    df["gender"] = np.select(
        [
            df["g_gender_nonconforming"] == 1,
            df["g_transgender_woman"] == 1,
            df["g_transgender_man"] == 1,
            df["g_female"] == 1,
            df["g_male"] == 1,
        ],
        [
            "Nonconforming",
            "Transgender Woman",
            "Transgender Man",
            "Female",
            "Male",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # REASON FOR CONTACT
    # ------------------------------------------------------------

    df["reason_for_contact"] = np.select(
        [
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 1),
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 2),
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 3),
            df["reason_for_stop"] == 2,
            df["reason_for_stop"] == 3,
            df["reason_for_stop"] == 4,
            df["reason_for_stop"] == 6,
            df["reason_for_stop"].isin([5, 7, 8]),
        ],
        [
            "Moving violation",
            "Equipment violation",
            "Non-moving violation",
            "Suspect criminal activity",
            "Parole/probation stop",
            "Outstanding arrest",
            "Consensual search",
            "Truancy/School Related",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # SUSPICION CATEGORY
    # First matching condition wins (like case_when in R)
    # ------------------------------------------------------------

    df["suspicion"] = np.select(
        [
            df["rfs_rs_off_witness"] == 1,
            df["rfs_rs_match_suspect"] == 1,
            df["rfs_rs_witness_id"] == 1,
            df["rfs_rs_carry_sus_object"] == 1,
            df["rfs_rs_actions_indicative"] == 1,
            df["rfs_rs_suspect_look"] == 1,
            df["rfs_rs_drug_trans"] == 1,
            df["rfs_rs_violent_crime"] == 1,
            df["rfs_rs_reason_susp"] == 1,
        ],
        [
            "Officer witnessed commission of a crime",
            "Matched suspect description",
            "Witness or victim ID of suspect at the scene",
            "Carrying suspicious object",
            "Actions indicative of casing a victim or location",
            "Suspected of acting as a lookout",
            "Actions indicative of a drug transaction",
            "Actions indicative of engaging in a violent crime",
            "Other reasonable suspicion of a crime",
        ],
        default=np.nan
    )

    # ------------------------------------------------------------
    # ANY SEARCH OCCURRED
    # True if either person search or property search occurred
    # ------------------------------------------------------------

    df["action_any_search"] = (
        (df["ads_search_person"] == 1) |
        (df["ads_search_property"] == 1)
    )

    # ------------------------------------------------------------
    # SEARCH BASES (convert 1/0 indicators into True/False)
    # ------------------------------------------------------------

    df["search_basis_plain_view"] = df["bfs_visible_contraband"] == 1
    df["search_basis_plain_smell"] = df["bfs_odor_contraband"] == 1
    df["search_basis_consent"] = df["bfs_consent_given"] == 1
    df["search_basis_safety"] = df["bfs_officer_safety"] == 1
    df["search_basis_suspect_weapon"] = df["bfs_suspect_weapon"] == 1
    df["search_basis_evidence_of_crime"] = df["bfs_evidence"] == 1
    df["search_basis_school_policy"] = df["bfs_school_policy"] == 1
    df["search_basis_emergency"] = df["bfs_exigent_circum"] == 1
    df["search_basis_canine"] = df["bfs_canine_detect"] == 1
    df["search_basis_warrant"] = df["bfs_search_warrant"] == 1
    df["search_basis_probation"] = df["bfs_parole"] == 1
    df["search_basis_incident_to_arrest"] = df["bfs_incident"] == 1
    df["search_basis_vehicle_inventory"] = df["bfs_vehicle_invent"] == 1

    # ------------------------------------------------------------
    # CONTRABAND VARIABLES
    # ------------------------------------------------------------

    # Weapons found if either weapon OR firearm indicator equals 1
    df["contraband_weapons"] = (
        (df["ced_weapon"] == 1) |
        (df["ced_firearm"] == 1)
    )

    # Any contraband found if "no contraband" is NOT equal to 1
    df["contraband_any"] = ~(df["ced_none_contraband"] == 1)

    # ------------------------------------------------------------
    # ARREST RESULT
    # ------------------------------------------------------------

    df["result_of_stop_arrest"] = (
        (df["ros_custodial_without_warrant"] == 1) |
        (df["ros_custodial_warrant"] == 1)
    )

    # Return a clean copy (prevents pandas fragmentation warnings)
    return df.copy()


In [ ]:
# ONLY DO THIS ONCE
# Load in the data
orange = load_ripa_from_doj_zip("Orange")

<positron-console-cell-3>:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2019 loaded successfully.


<positron-console-cell-3>:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2020 loaded successfully.


<positron-console-cell-3>:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


...

<positron-console-cell-3>:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2022 loaded successfully.


<positron-console-cell-3>:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2023 loaded successfully.


ImportError: `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.

In [ ]:
# Save the large dataset as a pickle so that don't need to load in again
orange.to_pickle("orange_2019_2023.pkl")

In [12]:
# Reload in data from parquet
orange = pd.read_pickle("orange_2019_2023.pkl")

In [ ]:
orange.head()

,doj_record_id,person_number,agency_ori,agency_name,time_of_stop,date_of_stop,stop_duration,closest_city,school_code,school_name,stop_student,k12_school_grounds,rae_full,rae_asian,rae_black_african_american,rae_hispanic_latino,rae_middle_eastern_south_asian,rae_native_american,rae_pacific_islander,rae_white,rae_multiracial,g_full,g_male,g_female,g_transgender_man,g_transgender_woman,g_gender_nonconforming,g_multigender,lgbt,age,age_group,limited_english_fluency,pd_full,pd_deafness_hearing,pd_speech_impair,pd_blind,pd_mental_health,pd_devel_disab,pd_hyperactivity_disability,pd_other,pd_none_disability,pd_disab_multi,reason_for_stop,rfs_traffic_violation_type,rfs_traffic_violation_code,rfs_rs_code,rfs_rs_off_witness,rfs_rs_match_suspect,rfs_rs_witness_id,rfs_rs_carry_sus_object,rfs_rs_actions_indicative,rfs_rs_suspect_look,rfs_rs_drug_trans,rfs_rs_violent_crime,rfs_rs_reason_susp,rfs_ec_discipline_code,rfs_ec_discipline,call_for_service,ads_removed_vehicle_order,ads_removed_vehicle_phycontact,ads_sobriety_test,ads_curb_detent,ads_handcuffed,ads_patcar_detent,ads_canine_search,ads_firearm_point,ads_firearm_discharge,ads_elect_device,ads_impact_discharge,ads_canine_bite,ads_baton,ads_chem_spray,ads_other_contact,ads_photo,ads_asked_search_per,ads_search_person,ads_asked_search_prop,ads_search_property,ads_prop_seize,ads_vehicle_impound,ads_written_statement,ads_no_actions,ads_search_pers_consen,ads_search_prop_consen,bfs_consent_given,bfs_officer_safety,bfs_search_warrant,bfs_parole,bfs_suspect_weapon,bfs_visible_contraband,bfs_odor_contraband,bfs_canine_detect,bfs_evidence,bfs_incident,bfs_exigent_circum,bfs_vehicle_invent,bfs_school_policy,ced_none_contraband,ced_firearm,ced_ammunition,ced_weapon,ced_drugs,ced_alcohol,ced_money,ced_drug_paraphernalia,ced_stolen_prop,ced_elect_device,ced_other_contraband,bps_safekeeping,bps_contraband,bps_evidence,bps_impound_vehicle,bps_abandon_prop,bps_violate_school,tps_firearm,tps_ammunition,tps_weapon,tps_drugs,tps_alcohol,tps_money,tps_drug_paraphernalia,tps_stolen_prop,tps_cellphone,tps_vehicle,tps_contraband,ros_no_action,ros_warning,ros_citation,ros_in_field_cite_release,ros_custodial_warrant,ros_custodial_without_warrant,ros_field_interview_card,ros_noncriminal_transport,ros_contact_legal_guardian,ros_psych_hold,ros_us_homeland,ros_referral_school_admin,ros_referral_school_counselor,ros_warning_cds,ros_citation_cds,ros_in_field_cite_release_cds,ros_custodial_wout_warrant_cds,year
0,W300021035GXD89HXE65,4,CA0300000,ORANGE CO SO,2125,2020-09-01,300,TUSTIN,NaN,NaN,0,0,2,0,1,0,0,0,0,0,0,2,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,2020
1,W30002104005BHMFLMO6,1,CA0300000,ORANGE CO SO,100,2020-12-16,120,LAKE FOREST,NaN,NaN,0,0,4,0,0,0,1,0,0,0,0,2,0,1,0,0,0,0,0,3,1,0,0,0,0,0,0,0,0,0,1,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,1,0,0,1,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,0,0,0,0,0,0,0,NaN,NaN,NaN,"32104, 35152, 35172, 35401, 35423",2020
2,W300021035G11SSXLRF0,1,CA0300000,ORANGE CO SO,2304,2020-08-26,20,STANTON,NaN,NaN,0,0,6,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,3,1,0,0,0,0,0,0,0,0,0,1,NaN,1,1.0,54106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,2020
3,W300021035GXD89HXE65,3,CA0300000,ORANGE CO SO,2125,2020-09-01,300,TUSTIN,NaN,NaN,0,0,2,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,4,1,1,0,0,0,0

In [13]:
# Checking to see the proportion of multiracial records
race_cols = [
    "rae_hispanic_latino",
    "rae_black_african_american",
    "rae_asian",
    "rae_middle_eastern_south_asian",
    "rae_pacific_islander",
    "rae_native_american",
    "rae_white",
]

race_counts = (orange[race_cols] == 1).sum(axis=1)
multi_rate = (race_counts > 1).mean()
print(f"Proportion multiracial: {multi_rate * 100:.2f}%")

Proportion multiracial: 1.19%


In [14]:
# Check if anyone is coded as more than 1 gender
gender_cols = [
    "g_gender_nonconforming",
    "g_transgender_woman",
    "g_transgender_man",
    "g_female",
    "g_male",
]

gender_counts_per_row = (orange[gender_cols] == 1).sum(axis=1)
multi_gender_rows = orange[gender_counts_per_row > 1]
prop_multi_gender = (gender_counts_per_row > 1).mean() * 100
print(f"Proportion multigender: {prop_multi_gender:.2f}%")

Proportion multigender: 0.05%
